# BERT for Spectrum

In [1]:
import torch as pt


mols_test = pt.load('./data/mine/test_11499.pt')
print(len(mols_test))
mols_all = pt.load('./data/mine/mols_all.pt')
print(len(mols_all))

11499
2253216


In [2]:
from torch.utils.data import DataLoader
from utils.data import SpecDataset, collate_fun_emb
import numpy as np

def collate_fun_bert(batch, min_len_mz:int=10, mask_prob:float=0.15, mask_token_id:int=0, spec_len:int=1000):
    # con: context, cen: center
    mzs, pad_masks = [], []
    max_len = max([len(mz) for mz, _ in batch])
    for mz, _ in batch:
        len_mz = len(mz)
        if len_mz >= min_len_mz: # 移除峰的数量小于阈值的质谱 
            pad_num = max_len - len_mz
            mzs.append(np.pad(mz, (0, pad_num), 'constant', constant_values=0))
            pad_mask = np.pad(pt.ones(len_mz), (0, pad_num), 'constant', constant_values=0)
            pad_masks.append(pad_mask)
    mzs = pt.tensor(np.array(mzs), dtype=pt.long)
    pad_masks = pt.tensor(np.array(pad_masks), dtype=pt.bool)
    labels = pt.full_like(mzs, -1, dtype=pt.long)
    # 生成随机掩码位置（15%）
    prob_matrix = pt.full(mzs.shape, mask_prob)
    masked_indices = pt.bernoulli(prob_matrix).bool()
    # 在padding位置不进行掩码
    masked_indices = masked_indices & pad_masks
    # 将原始token保存为标签（仅在被掩码的位置）
    labels[masked_indices] = mzs[masked_indices]
    # 80%替换为[MASK]
    indices_mask = pt.bernoulli(pt.full(mzs.shape, 0.8)).bool() & masked_indices
    mzs[indices_mask] = mask_token_id
    # 10%替换为随机词
    indices_random = (
        pt.bernoulli(pt.full(mzs.shape, 0.5)).bool() # 20% * 0.5 = 10%
        & masked_indices 
        & ~indices_mask # 在80%之外的20%
    )
    random_mzs = pt.randint(1, spec_len, mzs.shape, dtype=mzs.dtype)
    mzs[indices_random] = random_mzs[indices_random]
    # 剩余10%保持不变（不需要操作）
    # 取反是因为src_key_padding_mask是True表示padding位置
    return mzs, ~pad_masks, labels


dataset_lib = SpecDataset(mols_all)
dataset_test = SpecDataset(mols_test)
loader_lib = DataLoader(dataset_lib, batch_size=1024, shuffle=False,
                        num_workers=8, collate_fn=collate_fun_emb)
loader_test = DataLoader(dataset_test, batch_size=1024, shuffle=False,
                        num_workers=8, collate_fn=collate_fun_emb)
dataset_train = SpecDataset(dataset_lib, mapping=np.arange(232826))
loader_train = DataLoader(dataset_train, batch_size=256, shuffle=True,
                        num_workers=8, collate_fn=collate_fun_bert)

In [ ]:
import torch as pt
import torch.nn as nn
from torch.nn import TransformerEncoder, TransformerEncoderLayer


class Bertrum(nn.Module):
    def __init__(self, spec_len:int=1000, hidden_dim:int=512, num_heads:int=8, num_layers:int=1):
        super(Bertrum, self).__init__()
        self.spec_emb = nn.Embedding(spec_len, hidden_dim)
        self.layernorm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(0.1)

        encoder_layers = TransformerEncoderLayer(
            d_model=hidden_dim,
            nhead=num_heads,
            dim_feedforward=hidden_dim * 4,
            dropout=0.1,
            activation='gelu',
            batch_first=True,
        )
        self.encoder = TransformerEncoder(encoder_layers, num_layers=num_layers)
        self.mlm = nn.Linear(hidden_dim, spec_len)

    def _compute_embedding(self, data, power, emb_method:str='trans'):
        mzs, intens, masks = data  # [batch, seq]
        embs = self.spec_emb(mzs) # [batch, seq, emb_dim]
        embs = embs * masks.unsqueeze(-1)
        if emb_method == 'trans':
            embs = self.dropout(self.layernorm(embs))
            embs = self.encoder(embs, src_key_padding_mask=~masks)
        # intens [batch, seq]
        intens = pt.pow(intens, power)
        intens = intens.unsqueeze(-1)        
        embs = (embs * intens).sum(dim=1)
        return embs

    def forward(self, data:pt.Tensor, mode:str='train', power:float=0.5) -> pt.Tensor:
        if mode == 'train':
            # [batch_size, seq_len]
            mzs, pad_masks = data            
            # [batch_size, seq_len, hidden_dim]
            embs = self.spec_emb(mzs)
            embs = self.layernorm(embs)
            embs = self.dropout(embs)
            outputs = self.encoder(embs, src_key_padding_mask=pad_masks)
            outputs = self.mlm(outputs)
            return outputs
        elif mode == 'emb':            
            return self._compute_embedding(data, power)

In [4]:
from tqdm import tqdm
from utils.tools import gen_embeddings, build_idx, evaluate, save_model
from dataclasses import dataclass


@dataclass
class MLMConfig:
    spec_len: int = 1000
    epochs: int = 10
    gpu: int = 6
    model_name: str = 'bert'
    

class MLMTrainer:
    def __init__(self, model, config):
        super(MLMTrainer, self).__init__()
        self.model = model
        self.config = config
        self.f = open(f'{config.model_name}.txt', 'w')
        self.max_metrics = {'expand': [0, 0], 'insilico': [0, 0]}
        self.loss_fn = nn.CrossEntropyLoss(ignore_index=-1)
        self.optimizer = pt.optim.AdamW(self.model.parameters(), lr=1e-3)
        # self.scheduler = pt.optim.lr_scheduler.StepLR(self.optimizer, step_size=1, gamma=0.1)
    
    def compute_loss(self, data):
        # [batch_size, seq_len, spec_len]
        mzs, pad_masks, labels = data
        outputs = self.model((mzs, pad_masks)) 
        loss = self.loss_fn(outputs.view(-1, self.config.spec_len), labels.view(-1))
        return loss
    
    def train(self, loader_train, loader_test, loader_lib):
        for epoch in range(self.config.epochs):
            print(f'==================================Train_epoch{epoch+1}======================================')
            self.model.train()
            train_loss = []
            for i, batch in enumerate(tqdm(loader_train, unit='batch')):
                data = [b.to(self.config.gpu) for b in batch]
                self.optimizer.zero_grad()
                loss = self.compute_loss(data)
                train_loss.append(loss.item())
                loss.backward()
                self.optimizer.step()
                if (i+1) % 100 == 0:
                    loss = np.mean(train_loss)
                    print(f'Batch {i+1}, Loss: {loss:.4f}')
                    train_loss = []  
            # self.scheduler.step()
            print(f'===================================Test_epoch{epoch+1}======================================')
            self.f.write('\nTest_epoch%d\n' % (epoch+1))
            embeddings_lib = gen_embeddings(self.model, loader_lib, self.config.gpu)
            embeddings_test = gen_embeddings(self.model, loader_test, self.config.gpu)
            I_expand, _ = build_idx(embeddings_lib, embeddings_test, self.config.gpu)
            top1_expand, top10_expand = evaluate(mols_test, I_expand, mols_all, self.f, 'Expanded')
            if top1_expand > self.max_metrics['expand'][0] and top10_expand > self.max_metrics['expand'][1]:
                self.max_metrics['expand'] = [top1_expand, top10_expand]
                save_model(self.model, self.config.model_name, epoch)
            I_insilico, _ = build_idx(embeddings_lib[:2146690], embeddings_test, self.config.gpu)
            top1_insilico, top10_insilico = evaluate(mols_test, I_insilico, mols_all, self.f, 'In-silico')
            if top1_insilico > self.max_metrics['insilico'][0] and top10_insilico > self.max_metrics['insilico'][1]:
                self.max_metrics['insilico'] = [top1_insilico, top10_insilico]
                save_model(self.model, self.config.model_name, epoch)
            print(f'================================================================================================')
        self.f.close()

if __name__ == '__main__':
    config = MLMConfig()
    pt.cuda.set_device(config.gpu)
    model = Bertrum(config.spec_len).cuda(config.gpu)
    trainer = MLMTrainer(model, config)
    trainer.train(loader_train, loader_test, loader_lib)
            

==================================Train_epoch1======================================


 11%|█         | 101/910 [00:19<01:55,  7.00batch/s]

Batch 100, Loss: 5.2278


 22%|██▏       | 201/910 [00:35<01:58,  6.00batch/s]

Batch 200, Loss: 5.0287


...

100%|██████████| 910/910 [02:27<00:00,  6.16batch/s]

===================================Test_epoch2======================================


Searching time:  0:00:01.559586
Expanded library
Top1 hit rate: 37.66%
Top10 hit rate: 78.75%
Searching time:  0:00:01.494876
In-silico library
Top1 hit rate: 37.90%
Top10 hit rate: 79.09%
==================================Train_epoch3======================================


  6%|▌         | 52/910 [00:11<03:08,  4.55batch/s]


KeyboardInterrupt: 